# Dental Vision V1 — V3 visual error diagnosis
No retraining. Attach the successful V3 Kaggle output containing `dentex_holdout_v3.pt`. Rebuilds the exact split, then draws ground-truth vs predictions on held-out images and summarizes localization/class behavior.


In [ ]:
import os,shutil,pathlib,json,zipfile,glob,random
%cd /kaggle/working
shutil.rmtree('/kaggle/working/dental-vision-v1',ignore_errors=True)
!git clone https://github.com/drhaidarali95/dental-vision-v1.git /kaggle/working/dental-vision-v1
%cd /kaggle/working/dental-vision-v1
!pip -q install -r requirements.txt matplotlib


In [ ]:
cands=glob.glob('/kaggle/input/**/dentex_holdout_v3.pt',recursive=True)
print('CHECKPOINT CANDIDATES:',cands)
assert cands, 'Attach the successful V3 output containing dentex_holdout_v3.pt as Kaggle Input.'
ckpt=cands[0]; print('USING:',ckpt)


In [ ]:
root=pathlib.Path('data/dentex_visual'); shutil.rmtree(root,ignore_errors=True); root.mkdir(parents=True)
!python scripts/download_dentex.py --out data/dentex_visual --files training_data.zip
zpath=root/'training_data.zip'; wanted={'caries','deep caries','periapical lesion','periapical lesions','impacted','impacted tooth','impacted teeth'}; candidates=[]
with zipfile.ZipFile(zpath) as z:
    for name in z.namelist():
        if not name.lower().endswith('.json'): continue
        try: d0=json.loads(z.read(name))
        except Exception: continue
        if not isinstance(d0,dict) or not {'images','annotations'}.issubset(d0): continue
        cats=d0.get('categories_3') or d0.get('categories') or []; names={str(c.get('name','')).strip().lower() for c in cats}; score=len(names&wanted); bonus=2 if 'quadrant-enumeration-disease' in name.lower() else 0
        if score or bonus: candidates.append((score+bonus,len(d0['images']),name,d0,cats))
assert candidates
_,_,ann_member,d,cats=max(candidates,key=lambda x:(x[0],x[1])); d['categories']=cats
for a in d['annotations']:
    if 'category_id_3' in a: a['category_id']=a['category_id_3']
subset=root/'diagnostic'; (subset/'images').mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(zpath) as z:
    members=z.namelist()
    for im in d['images']:
        fn=str(im['file_name']).replace('\\','/').lstrip('./'); matches=[m for m in members if m.endswith('/'+fn) or m==fn] or [m for m in members if pathlib.PurePosixPath(m).name==pathlib.PurePosixPath(fn).name]; src=matches[0]; dest=subset/'images'/pathlib.PurePosixPath(fn).name
        with z.open(src) as r,open(dest,'wb') as w: shutil.copyfileobj(r,w)
        im['file_name']=dest.name
(subset/'all.json').write_text(json.dumps(d)); zpath.unlink()
!python scripts/split_dentex_holdout.py --annotations data/dentex_visual/diagnostic/all.json --train-out data/dentex_visual/diagnostic/train.json --val-out data/dentex_visual/diagnostic/val.json --val-fraction 0.20 --seed 20260920


In [ ]:
# Generate held-out GT/prediction overlays and machine-readable error summary.
import torch, matplotlib.pyplot as plt
from PIL import Image,ImageDraw
from torchvision.transforms import v2 as T
from train import build_model
from src.dentex_dataset import DentexCocoDataset
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ds=DentexCocoDataset('data/dentex_visual/diagnostic/images','data/dentex_visual/diagnostic/val.json',transforms=T.Compose([T.ToImage(),T.ToDtype(torch.float32,scale=True)]))
state=torch.load(ckpt,map_location=device,weights_only=False); model=build_model(state['num_classes']).to(device); model.load_state_dict(state['model']); model.eval()
outdir=pathlib.Path('/kaggle/working/v3_visual_diagnosis'); outdir.mkdir(exist_ok=True)
summary=[]
for i in range(len(ds)):
    x,t=ds[i]
    with torch.no_grad(): p=model([x.to(device)])[0]
    keep=p['scores'].detach().cpu()>=0.01; pb=p['boxes'].detach().cpu()[keep]; pl=p['labels'].detach().cpu()[keep]; ps=p['scores'].detach().cpu()[keep]
    info=ds.images[ds.ids[i]]; img=Image.open(pathlib.Path(ds.images_dir)/info['file_name']).convert('RGB'); draw=ImageDraw.Draw(img)
    for b,l in zip(t['boxes'],t['labels']):
        b=[float(v) for v in b]; draw.rectangle(b,outline='lime',width=4); draw.text((b[0],max(0,b[1]-14)),f'GT {int(l)}',fill='lime')
    for b,l,s in zip(pb,pl,ps):
        b=[float(v) for v in b]; draw.rectangle(b,outline='red',width=3); draw.text((b[0],b[1]),f'P {int(l)} {float(s):.2f}',fill='red')
    if i<40: img.save(outdir/f'{i:03d}_{info["file_name"]}.jpg')
    summary.append({'index':i,'file':info['file_name'],'gt_count':len(t['boxes']),'pred_count_thr001':len(pb),'top_scores':[round(float(v),4) for v in p['scores'].detach().cpu()[:10]],'gt_labels':[int(v) for v in t['labels']],'pred_labels_thr001':[int(v) for v in pl]})
json.dump(summary,open('/kaggle/working/v3_visual_error_summary.json','w'),indent=2)
print('images=',len(ds),'overlays=',min(40,len(ds)),'summary=/kaggle/working/v3_visual_error_summary.json')
print('GREEN=ground truth; RED=prediction; threshold=0.01')


In [ ]:
# Show first 12 overlays inline.
files=sorted(outdir.glob('*.jpg'))[:12]
for f in files:
    display(Image.open(f).resize((900,450)))
print('Send the screenshots/outputs plus v3_visual_error_summary.json to ChatGPT.')
